# SBC Grounding Study — Colab runner
Runs the harness against the corpus. Results persist to Google Drive so a runtime reset never loses trials (runner is resumable).

**Before running scored trials:** repo must be at tag `prereg-v1`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')   # add in Colab → Secrets
REPO = 'https://github.com/Phildram1/sbc-grounding-study.git'   # confirm name
TAG  = 'main'   # switch to 'prereg-v1' for the scored run

In [ ]:
%cd /content
!rm -rf sbc-grounding-study && git clone --branch $TAG $REPO
%cd sbc-grounding-study
!pip -q install -r requirements.txt
!git rev-parse HEAD

In [ ]:
# Guards must pass before any trial batch
!python -m pytest tests/ -q

In [ ]:
# Persist results on Drive
RESULTS = '/content/drive/MyDrive/sbc-grounding-study/results'
!mkdir -p $RESULTS
!rm -rf results && ln -s $RESULTS results
!python corpus/build_manifest.py

## Pilot — 5 configs × Arm A and B, 1 rep

In [ ]:
!python harness/run_trials.py --arm A --reps 1 --limit 5 --out results/pilot_A.jsonl
!python harness/run_trials.py --arm B --reps 1 --limit 5 --out results/pilot_B.jsonl

In [ ]:
!python scoring/score.py results/pilot_A.jsonl results/pilot_B.jsonl --labels labels/labels.json --out results/pilot_scores.json

## Arms C and E — require engine binding
Set `SBC_ENGINE_MODE` (`local_mcp` | `http` | `fixture`). Fixture mode is for plumbing only; the scorer refuses it without `--allow-fixture`.

In [ ]:
# os.environ['SBC_ENGINE_MODE'] = 'http'; os.environ['SBC_ENGINE_URL'] = '...'
# !python harness/run_trials.py --arm E --reps 1 --out results/E.jsonl
# !python harness/run_trials.py --arm C --reps 3 --out results/C.jsonl

## Full run (only at `prereg-v1`)

In [ ]:
# for arm in ['A','B','C']:
#     !python harness/run_trials.py --arm {arm} --out results/{arm}.jsonl
# !python harness/run_trials.py --arm E --out results/E.jsonl
# !python scoring/score.py results/A.jsonl results/B.jsonl results/C.jsonl results/E.jsonl --labels labels/labels.json --out results/scores.json
# !python analysis/stats.py results/scores.json --compare C B